# Gold layer — feature engineering

This notebook builds the Gold layer for the Kitsune SYN DoS anomaly detection
project. It reads from `kitsune_project.silver_layer.syn_dos_clean` and
produces model-ready datasets.

## Scope
This notebook covers two responsibilities:

1. **Feature selection** — evaluated reducing the 115 raw features using a
   discriminant score, but the score collapsed when recomputed on train only. Feature selection
   by score was dropped; all 115 features are kept, with selection deferred
   to the baseline model's own feature importances.
2. **Train/validation/test split** — split the dataset respecting
   chronological order. Random splitting is not valid here: the attack in
   this capture occurs within a specific time window rather than being
   distributed uniformly across the capture, so the split strategy depends
   on where that window falls relative to `row_id`.



## Locating the attack window and its internal distribution

Before designing the train/cv/test split, we need to know exactly where the
attack falls within the timeline (`row_id` range) and how it is distributed
inside that window. Random splitting is not valid here, since the attack is
concentrated in a specific time range rather than spread uniformly across
the capture.

Result: attacks are heavily front-loaded (74% fall within the first 3
deciles), tapering off toward the end of the capture. This shapes the split
strategy in the next section.

In [0]:
# Create the gold_layer schema if it doesn't exist yet
spark.sql("CREATE SCHEMA IF NOT EXISTS kitsune_project.gold_layer")

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("kitsune_project.silver_layer.syn_dos_clean")

# Get the first row_id, last row_id, and total count of attack rows
attack_row_id_range = (
    silver_df
    .filter(F.col("label") == 1)
    .agg(
        F.min("row_id").alias("first_attack_row_id"),
        F.max("row_id").alias("last_attack_row_id"),
        F.count("*").alias("attack_row_count")
    )
    .collect()[0]
)

# Store the boundaries in variables for later use
window_start = attack_row_id_range["first_attack_row_id"]
window_end = attack_row_id_range["last_attack_row_id"]
window_span = window_end - window_start

total_row_count = silver_df.count()

# Express the attack window as a percentage of the full timeline
first_pct = window_start / total_row_count * 100
last_pct = window_end / total_row_count * 100

print(f"Total rows: {total_row_count}")
print(f"First attack row_id: {window_start}")
print(f"Last attack row_id: {window_end}")
print(f"Attack row count: {attack_row_id_range['attack_row_count']}")
print(f"Attack window covers {first_pct:.2f}% to {last_pct:.2f}% of the capture")

In [0]:
# Keep only attack rows that fall inside the window we just found
attack_window_df = silver_df.filter(
    (F.col("row_id") >= window_start) & (F.col("label") == 1)
)

# Split the window into 10 equal chunks (deciles) and count attacks per chunk
attack_distribution = (
    attack_window_df
    .withColumn(
        "decile",
        ((F.col("row_id") - window_start) / window_span * 10).cast("int")
    )
    .groupBy("decile")
    .agg(F.count("*").alias("attack_count"))
    .orderBy("decile")
)

attack_distribution.show()

## Building the train, cv, and test splits

With the attack window and its internal distribution known, the splits are
defined as contiguous, chronologically ordered blocks based on percentiles
inside the attack window (`row_id`-based, no random shuffling, since a
random split would break temporal causality and let the model train on
future data).

Cutoffs used: 20% and 40% of the attack window span. This yields
train = deciles 0-1, cv = deciles 2-3, test = deciles 4-10 of the attack
window.

The 20%/40% cutoffs give cv and
test more attack examples,
which produces more statistically stable evaluation metrics. Train still retains a large number of attack examples (3,403), and class
imbalance within train is handled separately via scale_pos_weight, so the
reduced training volume is not a meaningful trade-off here.

The sanity check below confirms no split lost rows, and that every split
contains attack examples.

In [0]:
# compute the cut points for train, cv, and test based on percentiles inside the attack window
train_cv_cutoff = window_start + int(window_span * 0.2)  
cv_test_cutoff = window_start + int(window_span * 0.4)   

print(f"Train ends at row_id: {train_cv_cutoff}")
print(f"CV ends at row_id: {cv_test_cutoff}")

In [0]:
# Cell: apply the cut points to build train, cv, and test sets
train_df = silver_df.filter(F.col("row_id") < train_cv_cutoff)
cv_df = silver_df.filter(
    (F.col("row_id") >= train_cv_cutoff) & (F.col("row_id") < cv_test_cutoff)
)
test_df = silver_df.filter(F.col("row_id") >= cv_test_cutoff)

# Sanity check: confirm each split has attack examples 
for name, df in [("train", train_df), ("cv", cv_df), ("test", test_df)]:
    total = df.count()
    attacks = df.filter(F.col("label") == 1).count()
    print(f"{name}: {total} rows, {attacks} attacks")

## Recomputing feature scores on train only

The discriminant scores from `03_EDA` were calculated over the full dataset. Now that train/cv/test exist, that approach is a form
of leakage: feature selection would be influenced by rows that belong to cv
and test, which should have no effect on any decision made before training.

The cell below recomputes the same score formula
(`|mean(attack) - mean(normal)| / stddev(normal)`) using only `train_df`.

Result: the ranking changes completely. None of the previously top-scoring
features (`feature_79`, `feature_76`, etc.) appear in the new top 15, and no
feature exceeds a score of 4 (the threshold that looked like a natural cut
in the full-dataset ranking). This score, computed on train alone, does not
provide a usable feature-selection criterion for this project.

In [0]:
from pyspark.sql import functions as F
import pandas as pd

feature_columns = [c for c in silver_df.columns if c.startswith("feature_")]

# Build one mean + stddev aggregation per feature, computed only on train_df
agg_exprs = []
for feature in feature_columns:
    agg_exprs.append(F.mean(feature).alias(f"{feature}_mean"))
    agg_exprs.append(F.stddev(feature).alias(f"{feature}_stddev"))

class_stats_pd = train_df.groupBy("label").agg(*agg_exprs).toPandas()

normal_stats = class_stats_pd[class_stats_pd["label"] == 0].iloc[0]
attack_stats = class_stats_pd[class_stats_pd["label"] == 1].iloc[0]

# Same formula as EDA
scores = {}
for feature in feature_columns:
    mean_diff = abs(attack_stats[f"{feature}_mean"] - normal_stats[f"{feature}_mean"])
    stddev_normal = normal_stats[f"{feature}_stddev"]
    scores[feature] = mean_diff / stddev_normal if stddev_normal and stddev_normal > 0 else 0.0

scores_df = (
    pd.DataFrame(scores.items(), columns=["feature", "score"])
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

print(scores_df.head(15))

## Why did the scores collapse?

This cell checks if `feature_79` (previously the top-scoring feature)
behaves differently across the attack window over time.

Result: `feature_79` stays flat and normal-looking through deciles 0-2,
then jumps by about 11 orders of magnitude from decile 3 onward. Train
(deciles 0-1) only sees the quiet start of the attack — the strong signal
lives entirely in cv and test.

This explains the score collapse: it's a real pattern in the data, not a
bug. A simple mean-difference score can't catch a signal this non-linear.
Feature selection by score is dropped — the baseline model will use all features, and we'll check its own
feature importances after training.

Known limitation: since train only has the "quiet" phase of the attack,
cv/test metrics may look better than what the model could actually do on
an attack in its earliest moments.

In [0]:
# Verify: does feature_79 show extreme values concentrated in later deciles?
# Reuses attack_window_df and window_start/window_span from earlier cells
check_df = (
    attack_window_df
    .withColumn(
        "decile",
        ((F.col("row_id") - window_start) / window_span * 10).cast("int")
    )
    .groupBy("decile")
    .agg(
        F.mean("feature_79").alias("mean_feature_79"),
        F.max("feature_79").alias("max_feature_79")
    )
    .orderBy("decile")
)

check_df.show()

## Class balance and saving the Gold tables

Before saving, we check the exact ratio of attacks to normal traffic in
each split, to have a documented number instead of just knowing it's
"imbalanced".

Then the three splits are written as Gold Delta tables, and tagged with
table properties (split method, cutoffs, imbalance strategy) so anyone
who opens these tables later can see how they were built without needing
this notebook's history.

In [0]:
# Quantify class balance in each split, and document the decision
for name, df in [("train", train_df), ("cv", cv_df), ("test", test_df)]:
    total = df.count()
    attacks = df.filter(F.col("label") == 1).count()
    attack_pct = attacks / total * 100
    print(f"{name}: {attacks} attacks / {total} rows ({attack_pct:.3f}% positive)")

Note: positive rates differ a lot across splits — train 0.13%, cv 15.0%,
test 1.8% — none matches the real production rate (0.25%). This happens
because attack density isn't even across the timeline. Read cv scores with caution — cv is much easier than production or even
test, since it has way more attack examples to work with.

In [0]:
# Save train, cv, and test splits as Gold Delta tables
train_df.write.format("delta").mode("overwrite").saveAsTable(
    "kitsune_project.gold_layer.syn_dos_train"
)
cv_df.write.format("delta").mode("overwrite").saveAsTable(
    "kitsune_project.gold_layer.syn_dos_cv"
)
test_df.write.format("delta").mode("overwrite").saveAsTable(
    "kitsune_project.gold_layer.syn_dos_test"
)

print("Gold tables written: syn_dos_train, syn_dos_cv, syn_dos_test")

In [0]:
# Attach split parameters as table properties, for reproducibility
for table_name in ["syn_dos_train", "syn_dos_cv", "syn_dos_test"]:
    spark.sql(f"""
        ALTER TABLE kitsune_project.gold_layer.{table_name}
        SET TBLPROPERTIES (
            'split_method' = 'chronological, based on row_id percentiles inside attack window',
            'train_cv_cutoff_row_id' = '{train_cv_cutoff}',
            'cv_test_cutoff_row_id' = '{cv_test_cutoff}',
            'class_imbalance_strategy' = 'scale_pos_weight, computed per model on train only'
        )
    """)

print("Table properties set on all three Gold tables")

## Summary

This notebook prepared the Gold layer for modeling:

- **Attack window found**: attacks fall in the last 3.21% of the capture
  (row_id 2,682,349–2,771,275), heavily front-loaded within that window.
- **Feature selection dropped**: the discriminant score collapsed once
  recomputed on train only, so all 115 features are kept — the baseline
  model will decide what matters via its own feature importances.
- **Chronological split built**: train (deciles 0-1), cv (deciles 2-3),
  test (deciles 4-10) of the attack window — no random shuffling, to avoid
  training on future data.
- **Class balance documented**: positive rate differs sharply across splits
  (train 0.13%, cv 15.0%, test 1.8%) — none matches real production
  imbalance (0.25%). CV scores should be read with that in mind.
- **Output**: `syn_dos_train`, `syn_dos_cv`, `syn_dos_test` saved to
  `kitsune_project.gold_layer`, tagged with split method and cutoffs as
  table properties.

